# Obtaining Data

In [ ]:
from astroquery.ipac.irsa import Irsa
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt
from matplotlib import rcParams

In [ ]:
light_df = subset_1_5k[['objID', 'raMean', 'decMean']]

coords = SkyCoord(ra=light_df['raMean'].values*u.deg, dec=light_df['decMean'].values*u.deg, frame='icrs')
radius = 1.5 * u.arcsec
columns = ['ra', 'dec', 'w1mpro', 'w2mpro', 'w1sigmpro', 'w2sigmpro', 'mjd']

wise_resultsC1 = []

def query_wise(galaxy_name, coord):
    """Query NEOWISE-R for a single galaxy."""
    try:
        result = Irsa.query_region(
            coord,
            catalog='neowiser_p1bs_psd',
            spatial='Cone',
            radius=radius,
        )
        if len(result) == 0:
            print(f"No WISE data found for {galaxy_name}")
            return None
        df = result.to_pandas()[columns]
        df['Name'] = galaxy_name
        return df
    except Exception as e:
        print(f"Error querying {galaxy_name}: {e}")
        return None

In [ ]:
# Parallel execution — significantly reduces query time for large source lists
with ThreadPoolExecutor(max_workers=10) as executor:  # adjust max_workers for your CPU/network
    futures = [executor.submit(query_wise, light_df.iloc[i]['objID'], coord) for i, coord in enumerate(coords)]
    
    for future in as_completed(futures):
        df = future.result()
        if df is not None:
            wise_resultsC1.append(df)

#combine
wise_dfqueryC1 = pd.concat(wise_resultsC1, ignore_index=True) if wise_resultsC1 else pd.DataFrame()

print(wise_dfqueryC1)
wise_dfqueryC1.to_csv("wise_results_1_5k.csv", index=False)

# Cleaning Data

In [ ]:
INPUT_CSV  = "wise_results_combined1.csv"
OUTPUT_CSV = "cleaned_binned_lightcurvesQ1.csv"

EPOCH_LENGTH = 180.0   # dps (NEOWISE cadence)
MAD_THRESHOLD = 3.0    # outlier cutoff



# MAD OUTLIER FILTER
def mad_filter(values, threshold=MAD_THRESHOLD):
    values = np.asarray(values)
    median = np.nanmedian(values)
    mad = np.nanmedian(np.abs(values - median))

    if mad == 0 or np.isnan(mad):
        return np.ones(len(values), dtype=bool)

    return np.abs(values - median) <= threshold * mad



# MAIN CLEANING FUNCTION
def clean_and_bin(df, mag_col, err_col, band_name):
    results = []

    for name, group in df.groupby("Name"):
        mjd_min = group["mjd"].min()
        group = group.copy()

        # Assign NEOWISE epoch index
        group["epoch"] = np.floor((group["mjd"] - mjd_min) / EPOCH_LENGTH)

        for ep, ep_group in group.groupby("epoch"):
            mags = ep_group[mag_col].values
            errs = ep_group[err_col].values if err_col in ep_group else None

            valid = ~np.isnan(mags)
            mags = mags[valid]
            errs = errs[valid] if errs is not None else None

            if len(mags) == 0:
                continue

            # Outlier mask (computed on mags)
            keep = mad_filter(mags)

            mags_clean = mags[keep]
            errs_clean = errs[keep] if errs is not None else None

            if len(mags_clean) == 0:
                continue

            # Mean magnitude
            mag_mean = np.mean(mags_clean)
            mag_std = np.std(mags_clean)
            mjd_mean = ep_group["mjd"].mean()


            results.append({
                "Name": name,
                "band": band_name,
                "epoch": int(ep),
                "mjd_mean": mjd_mean,
                "mag_mean": mag_mean,
                "mag_err_mean": mag_std,
                "n_raw": len(mags),
                "n_used": len(mags_clean),
                #"n_err_used": n_err
            })

    return pd.DataFrame(results)


# LOAD DATA
df = pd.read_csv(INPUT_CSV)

# PROCESS W1 AND W2
w1_df = df[["Name", "mjd", "w1mpro", "w1sigmpro"]].copy()
w2_df = df[["Name", "mjd", "w2mpro", "w2sigmpro"]].copy()

w1_clean = clean_and_bin(w1_df, "w1mpro", "w1sigmpro", "W1")
w2_clean = clean_and_bin(w2_df, "w2mpro", "w2sigmpro", "W2")

# COMBINE & SAVE
cleaned = pd.concat([w1_clean, w2_clean], ignore_index=True)
cleaned.to_csv(OUTPUT_CSV, index=False)

print(f"Saved cleaned, epoch-binned light curves to {OUTPUT_CSV}")

In [ ]:
cb = pd.read_csv("cleaned_binned_lightcurvesQ1.csv")

w1_df = cb[cb['band'] == 'W1'].copy()

# Keep only bins with n_used >= 3 per source
# Actually, since n_used is already per bin, we can just filter directly
w1_filtered = w1_df[w1_df['n_used'] >= 3].copy()

# Reset index for cleanliness
w1_filtered.reset_index(drop=True, inplace=True)

# Save to a new CSV
w1_filtered.to_csv('Q1filtered.csv', index=False)

print(f"Original W1 rows: {len(w1_df)}")
print(f"Filtered W1 rows: {len(w1_filtered)}")

In [ ]:
orig = pd.read_csv("wise_results_combined1.csv")
var_Q1 = var_Q1.copy()
lc_Q1 = lc_Q1.copy()

orig["Name"] = orig["Name"].astype("Int64").astype("string")
lc_Q1["Name"] = lc_Q1["Name"].astype("Int64").astype("string")
var_Q1["Name"] = var_Q1["Name"].astype("Int64").astype("string")

In [ ]:
coords = orig.groupby("Name")[["ra", "dec"]].first().reset_index()

lc_Q1 = lc_Q1.merge(coords, on="Name", how="left")

# Variability Filtering

In [ ]:
df = pd.read_csv("Q1filtered.csv")

results = []

for (name, band), g in df.groupby(["Name", "band"]):

    mags = g["mag_mean"].to_numpy()
    errs = g["mag_err_mean"].to_numpy()

    finite = np.isfinite(mags) & np.isfinite(errs)
    mags = mags[finite]
    errs = errs[finite]

    N = mags.size
    if N < 3:
        continue

    mean_mag = np.mean(mags)

    S2 = np.var(mags, ddof=1)
    mean_err2 = np.mean(errs**2)

    xs_var = S2 - mean_err2
    nxs_var = xs_var / mean_mag**2
        
    if xs_var > 0:
        err_nxs = np.sqrt(
            (np.sqrt(2/N) * mean_err2 / mean_mag**2)**2 +
            (np.sqrt(mean_err2/N) * (2*np.sqrt(xs_var)) / mean_mag**2)**2
        )
    else:
        err_nxs = np.nan

    results.append({
        "Name": name,
        "band": band,
        "N_epochs": N,
        "mean_mag": mean_mag,
        "excess_var": xs_var,
        "norm_excess_var": nxs_var,
        "norm_excess_var_err": err_nxs,
        "variability_significance": nxs_var / err_nxs if err_nxs > 0 else 0.0
    
    })

xs_Q1 = pd.DataFrame(results)

In [ ]:
variable = xs_Q1["variability_significance"] >= 3
# How many unique galaxies with >= 3sigma variability 
variable_names = xs_Q1.loc[
    xs_Q1["variability_significance"] >= 3, "Name"
].unique()

len(variable_names)

In [ ]:
# Load data
lc_Q1 = pd.read_csv("cleaned_binned_lightcurvesQ1.csv")

# Variable light curves (>= 3 sigma)
var_Q1 = xs_Q1[xs_Q1["variability_significance"] >= 3]

# Lightcurves with Variability

In [ ]:
for name in var_Q1["Name"].unique():

    g = lc_Q1.loc[lc_Q1["Name"] == name]
    if g.empty:
        continue

    gb = g.loc[g["band"] == "W1"]
    if gb.empty:
        continue

    # --- get RA/Dec ---
    row = lc_Q1.loc[lc_Q1["Name"] == name].iloc[0]
    ra = row["ra"]
    dec = row["dec"]

    plt.figure(figsize=(7, 4))

    plt.errorbar(
        gb["mjd_mean"],
        gb["mag_mean"],
        yerr=gb["mag_err_mean"],
        fmt="o",
        ms=8,
        lw=2,
        capsize=4,
        alpha=0.8,
        label="W1",
    )

    plt.gca().invert_yaxis()
    plt.xlabel("MJD", fontsize=18)
    plt.ylabel("Magnitude", fontsize=18)

    plt.title(f"{name}\nRA={ra:.5f}, Dec={dec:.5f}", fontsize=16)

    plt.tick_params(axis="both", which="major", labelsize=14)
    plt.grid(alpha=0.3)
    plt.legend(fontsize=12)
    plt.tight_layout()

    plt.show()
    plt.close()